In [1]:
import sys, os

%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%%RecordEvent
# %load_ext cudf.pandas
import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [3]:
import time

start_time = time.perf_counter()

In [4]:
### cell 0 ###

# load & cleanup
file = "/home/dias-benchmarks/notebooks/kkhandekar/environmental-vs-ai-startups-india-eda/input/indian-startup-recognized-by-dpiit/Startup_Counts_Across_India.csv"
df = pd.read_csv(file)

# -- STEFANOS -- Replicate Data

In [5]:
### cell 1 ###

factor = 3000
df = pd.concat([df] * factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17673000 entries, 0 to 5890
Data columns (total 5 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   S No.     int64 
 1   Year      int64 
 2   State     object
 3   Industry  object
 4   Count     int64 
dtypes: int64(3), object(2)
memory usage: 809.0+ MB


In [6]:
### cell 2 ###

df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

,Year,State,Industry,Count
0,2022,Andaman and Nicobar Islands,Agriculture,1
1,2022,Andaman and Nicobar Islands,AR VR (Augmented + Virtual Reality),1
2,2022,Andaman and Nicobar Islands,Construction,1
3,2022,Andaman and Nicobar Islands,Internet of Things,1
4,2022,Andaman and Nicobar Islands,Marketing,1


In [7]:
### cell 3 ###

### Optimized Code ###

# Define industry categories for filtering
env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai = ["AI", "Robotics", "Computer Vision"]

# Create a combined dataframe for environmental and AI startups
industry_map = {ind: "ENV" for ind in env}
industry_map.update({ind: "AI" for ind in ai})

df_ea = df[df["Industry"].map(industry_map).notna()].copy()
df_ea["MainIndustry"] = df_ea["Industry"].map(industry_map)

# Calculate basic statistics
main_industry_counts = df_ea["MainIndustry"].value_counts()
print(
    f"A total of {df_ea.shape[0]} startups were started in India between 2016 & 2022, out of which {main_industry_counts.get('ENV', 0)} are environmental related & {main_industry_counts.get('AI', 0)} are AI startups."
)

A total of 2361000 startups were started in India between 2016 & 2022, out of which 1473000 are environmental related & 888000 are AI startups.


In [8]:
end_time = time.perf_counter()
print(end_time - start_time)

4.994812449000165
